### Ant Colony Optimization (ACO) for Sudoku

Now, let's define the core functions for the Ant Colony Optimization algorithm: `is_valid_placement`, `initialize_pheromones`, `calculate_desirability`, `ant_construct_solution`, `update_pheromones`, and the `ant_colony_optimization` itself.

### Sudoku Boards Definitions

In [ ]:
sudoku_4x4 = [
    [1, 2, 0, 0],
    [3, 4, 0, 0],
    [0, 0, 1, 2],
    [0, 0, 3, 4]
]

sudoku_9x9 = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

In [ ]:
import numpy as np
import pandas as pd
import random
from copy import deepcopy
from tqdm.notebook import trange

# Define the fitness function (assuming it's defined elsewhere, but good to ensure it exists)
def fitness(board, N):
    # This is a placeholder. A real fitness function would count conflicts.
    # For this context, we assume a solution is valid if there are no conflicts.
    conflicts = 0
    sqrt_N = int(np.sqrt(N))

    for r in range(N):
        # Check rows
        row_values = board[r, :]
        conflicts += len(row_values) - len(np.unique(row_values[row_values != 0]))

    for c in range(N):
        # Check columns
        col_values = board[:, c]
        conflicts += len(col_values) - len(np.unique(col_values[col_values != 0]))

    for r_block in range(sqrt_N):
        for c_block in range(sqrt_N):
            block_values = board[r_block * sqrt_N:(r_block + 1) * sqrt_N, c_block * sqrt_N:(c_block + 1) * sqrt_N].flatten()
            conflicts += len(block_values) - len(np.unique(block_values[block_values != 0]))
    return conflicts

def get_fixed_positions(board, N):
    fixed = []
    for r in range(N):
        for c in range(N):
            if board[r, c] != 0:
                fixed.append((r, c))
    return fixed


# Parameters for ACO 4x4 Sudoku
num_ants_aco_4x4 = [10, 20]
evaporation_rates_aco_4x4 = [0.1, 0.3]
Q_values_aco_4x4 = [0.5, 1.0]
max_generations_aco_4x4 = 500

# Parameters for ACO 9x9 Sudoku
num_ants_aco_9x9 = [20, 50]
evaporation_rates_aco_9x9 = [0.1, 0.3]
Q_values_aco_9x9 = [0.5, 1.0]
max_generations_aco_9x9 = 1000

repeats_aco = 1 # Number of times to repeat each experiment

In [ ]:
def is_valid_placement(board, row, col, num, N):
    """
    Checks if placing 'num' at (row, col) on the 'board' is valid
    according to Sudoku rules (no duplicates in row, column, or 3x3 subgrid).

    Args:
        board (np.array): The current Sudoku board (partial or complete).
        row (int): The row index.
        col (int): The column index.
        num (int): The number to check for placement.
        N (int): The size of the Sudoku grid (e.g., 4 for 4x4, 9 for 9x9).

    Returns:
        bool: True if the placement is valid, False otherwise.
    """
    # Check row: If 'num' already exists in the given row, it's not valid.
    if num in board[row, :]:
        return False

    # Check column: If 'num' already exists in the given column, it's not valid.
    if num in board[:, col]:
        return False

    # Check 3x3 (or sqrt_N x sqrt_N) subgrid:
    # Calculate the size of the subgrid (e.g., 2 for 4x4, 3 for 9x9).
    sqrt_N = int(np.sqrt(N))

    # Determine the starting row and column of the subgrid that contains (row, col).
    start_row, start_col = sqrt_N * (row // sqrt_N), sqrt_N * (col // sqrt_N)

    # Check if 'num' already exists within this subgrid.
    if num in board[start_row:start_row + sqrt_N, start_col:start_col + sqrt_N]:
        return False

    # If all checks pass, the placement is valid.
    return True

def initialize_pheromones(N, initial_pheromone=0.1):
    """
    Initializes the pheromone matrix for all possible placements.

    Args:
        N (int): The size of the Sudoku grid.
        initial_pheromone (float): The initial pheromone value for all cells.

    Returns:
        np.array: A 3D NumPy array representing the pheromone matrix.
                  pheromones[r, c, num] stores the pheromone level for placing 'num' at (r, c).
    """
    # Create a 3D array: (rows, columns, values 1 to N+1). N+1 allows 1-based indexing for values.
    # All elements are initialized to 'initial_pheromone'.
    return np.full((N, N, N + 1), initial_pheromone, dtype=float)

def calculate_desirability(pheromones, r, c, num, N, alpha=1.0, beta=1.0):
    """
    Calculates the desirability of placing 'num' at (r, c).
    In this simplified ACO, desirability is primarily based on pheromone levels.

    Args:
        pheromones (np.array): The pheromone matrix.
        r (int): Row index.
        c (int): Column index.
        num (int): The number being considered.
        N (int): Size of the Sudoku grid.
        alpha (float): Exponent for pheromone importance.
        beta (float): Exponent for heuristic importance (not fully utilized here).

    Returns:
        float: The desirability value.
    """
    # Pheromone contribution: Pheromone level for placing 'num' at (r, c) raised to 'alpha'.
    phi = pheromones[r, c, num]**alpha

    # Heuristic contribution (eta):
    # The comment suggests a more complex heuristic could be used. Currently, 'eta' is not explicitly used.
    # For Sudoku, a heuristic might consider how many conflicts a number would create immediately,
    # or how many empty cells it would block.
    # For this implementation, the validity check in 'ant_construct_solution' acts as a strong heuristic.

    # Currently, only pheromone is returned. Heuristic will be implicitly handled by 'ant_construct_solution'.
    return phi

def ant_construct_solution(board_template, fixed_pos, pheromones, N):
    """
    An individual ant constructs a Sudoku solution.

    Args:
        board_template (list of lists): The initial Sudoku board with pre-filled numbers.
        fixed_pos (list of tuples): List of (row, col) for pre-filled cells (not used directly in this version).
        pheromones (np.array): The current pheromone matrix.
        N (int): The size of the Sudoku grid.

    Returns:
        tuple: A tuple containing:
               - current_board (np.array): The Sudoku board constructed by this ant.
               - choices (list): A list of (row, col, chosen_num) tuples representing the ant's choices.
    """
    # Create a deep copy of the initial board for the ant to work on.
    current_board = deepcopy(board_template)

    # Identify all empty cells (where value is 0).
    empty_cells = [(r, c) for r in range(N) for c in range(N) if board_template[r][c] == 0]

    # Randomize the order in which the ant fills empty cells. This adds stochasticity.
    random.shuffle(empty_cells)

    # List to store the choices made by this ant (used for pheromone updates).
    choices = [] # Stores (row, col, chosen_num)

    # Iterate through each empty cell the ant needs to fill.
    for r, c in empty_cells:
        possible_values = [] # Numbers 1-N that can be placed in (r,c) validly.
        probabilities = []   # Corresponding probabilities (based on pheromones).

        # For each possible number (1 to N):
        for num in range(1, N + 1):
            # Check if placing 'num' is valid based on the current state of the ant's board.
            if is_valid_placement(current_board, r, c, num, N):
                possible_values.append(num)
                # Add the pheromone level for this valid placement to the probabilities list.
                probabilities.append(pheromones[r, c, num])

        # If no valid numbers can be placed in the current cell (ant is stuck):
        if not possible_values:
            # The ant 'fails' at this cell. A random number is placed to complete the board.
            # This path will likely have a high fitness (many conflicts) and thus low pheromone deposition.
            current_board[r, c] = random.randint(1, N)
            choices.append((r, c, current_board[r, c]))
            continue # Move to the next empty cell.

        # Normalize probabilities for selection.
        total_prob = sum(probabilities)
        if total_prob == 0: # If all pheromone levels are zero for valid moves (e.g., at start or after strong evaporation).
            # Choose randomly among the valid moves.
            chosen_num = random.choice(possible_values)
        else:
            # Normalize probabilities so they sum to 1.
            probabilities = [p / total_prob for p in probabilities]
            # Select a number probabilistically based on the normalized pheromone levels.
            chosen_num = random.choices(possible_values, weights=probabilities, k=1)[0]

        # Place the chosen number on the ant's current board.
        current_board[r, c] = chosen_num
        # Record the choice for later pheromone updates.
        choices.append((r, c, chosen_num))

    return current_board, choices

def update_pheromones(pheromones, best_solution, initial_board, N, evaporation_rate, Q):
    """
    Updates the pheromone trails based on the best solution found in a generation.

    Args:
        pheromones (np.array): The current pheromone matrix.
        best_solution (np.array): The best Sudoku solution found by an ant in the current iteration.
        initial_board (list of lists): The original Sudoku board template (to identify fixed cells).
        N (int): The size of the Sudoku grid.
        evaporation_rate (float): The rate at which pheromones evaporate.
        Q (float): Pheromone deposition constant.
    """
    # 1. Evaporation: All pheromone trails decrease over time.
    # This simulates pheromones naturally dissipating and helps to forget poor paths.
    pheromones *= (1 - evaporation_rate)

    # 2. Deposition by best ant:
    # Calculate the fitness of the best solution to determine how much pheromone to deposit.
    # Lower fitness (fewer conflicts) means a better solution.
    best_fitness = fitness(best_solution, N)

    # Determine the amount of pheromone to deposit. This is typically inversely proportional
    # to the fitness of the solution (lower fitness -> more deposition).
    if best_fitness == 0:
        # If a perfect solution is found, deposit a large, constant amount of pheromone (Q).
        delta_pheromone = Q
    else:
        # Otherwise, deposit pheromone inversely proportional to the fitness.
        # Adding a small epsilon (1e-6) prevents division by zero if fitness somehow becomes 0 but isn't solved.
        delta_pheromone = Q / (best_fitness + 1e-6)

    # Deposit pheromones along the path taken by the 'best_solution'.
    for r in range(N):
        for c in range(N):
            # Pheromones are only deposited on cells that were initially empty
            # in the original puzzle (i.e., cells an ant actually had to decide upon).
            if initial_board[r][c] == 0:
                chosen_num = best_solution[r, c]
                # Ensure a number was actually placed in this cell (i.e., not 0 if the board was incomplete).
                if chosen_num != 0:
                    # Add 'delta_pheromone' to the specific pheromone trail for this choice.
                    pheromones[r, c, chosen_num] += delta_pheromone

def ant_colony_optimization(board, N, generations, num_ants, evaporation_rate, Q, initial_pheromone=0.1):
    """
    Main function for Ant Colony Optimization to solve Sudoku.

    Args:
        board (list of lists): The initial Sudoku board to solve.
        N (int): The size of the Sudoku grid.
        generations (int): The maximum number of iterations.
        num_ants (int): The number of ants to simulate in each generation.
        evaporation_rate (float): The rate at which pheromones evaporate.
        Q (float): Pheromone deposition constant.
        initial_pheromone (float): Initial value for all pheromones.

    Returns:
        tuple: A tuple containing:
               - solved (bool): True if a solution was found (fitness 0), False otherwise.
               - generations_taken (int): The number of generations it took to find a solution.
               - best_overall_solution (np.array): The best Sudoku board found.
    """
    # 1. Initialization:
    # Initialize the pheromone matrix.
    pheromones = initialize_pheromones(N, initial_pheromone)

    # Keep track of the best solution found across all generations.
    best_overall_solution = None
    best_overall_fitness = float('inf') # Initialize with a very high fitness (bad).

    # Identify fixed positions (pre-filled cells) that ants cannot change.
    fixed_pos = get_fixed_positions(board, N)

    # Convert the initial board to a NumPy array for consistent operations.
    np_board = np.array(board) # Ensure board is a numpy array for slicing

    # 2. Main Loop (Generations):
    # Iterate for a specified number of generations.
    for gen in trange(generations, desc=f"ACO N={N}, Ants={num_ants}, Evap={evaporation_rate}", leave=False):
        # In each generation, keep track of the best solution found by any ant in this generation.
        current_best_ant_solution = None
        current_best_ant_fitness = float('inf')

        # 3. Ant Construction Phase:
        # Each ant constructs a candidate solution.
        for _ in range(num_ants):
            # Call 'ant_construct_solution' to get a board built by an ant and its choices.
            ant_solution, ant_choices = ant_construct_solution(np_board, fixed_pos, pheromones, N)

            # Evaluate the fitness (number of conflicts) of the ant's solution.
            ant_fitness = fitness(ant_solution, N)

            # Update the best solution found in the current generation if this ant's solution is better.
            if ant_fitness < current_best_ant_fitness:
                current_best_ant_fitness = ant_fitness
                current_best_ant_solution = ant_solution

        # 4. Global Best Update:
        # Check if the best solution from this generation is better than the overall best found so far.
        if current_best_ant_fitness < best_overall_fitness:
            best_overall_fitness = current_best_ant_fitness
            best_overall_solution = current_best_ant_solution
            # If a perfect solution (fitness 0) is found, terminate early.
            if best_overall_fitness == 0:
                print(f"Solved by ACO in generation {gen}!")
                # Return True, the number of generations taken, and the solved board.
                return True, gen, best_overall_solution

        # 5. Pheromone Update Phase:
        # Update pheromones based on the overall best solution found *so far* (or current generation's best).
        # Using best_overall_solution ensures that good paths are consistently reinforced.
        if best_overall_solution is not None:
            # Pass the original board (template) to update_pheromones to correctly identify mutable cells.
            update_pheromones(pheromones, best_overall_solution, board, N, evaporation_rate, Q)

    # If the loop finishes without finding a perfect solution:
    # Return False (not solved), the total generations, and the best board found.
    return False, generations, best_overall_solution

In [ ]:
def run_experiment_aco(sudoku_board, N, num_ants_list, evaporation_rates, Q_values, repeats=1, max_generations=1000):
    results = []

    for num_ants in num_ants_list:
        for evap_rate in evaporation_rates:
            for Q_val in Q_values:
                success_count = 0
                total_generations_for_solved = 0 # To track generations for successful runs
                best_overall_board = None # Best board across all repeats for current params
                best_overall_fitness = float('inf') # Fitness of best_overall_board

                for r in trange(repeats, desc=f"ACO N={N}, Ants={num_ants}, Evap={evap_rate}, Q={Q_val}", leave=False):
                    solved, gens, final_board = ant_colony_optimization(
                        board=sudoku_board, N=N, generations=max_generations,
                        num_ants=num_ants, evaporation_rate=evap_rate, Q=Q_val
                    )

                    if solved:
                        success_count += 1
                        total_generations_for_solved += (gens + 1) # Add generations taken for this successful run
                        print(f"Solved N={N} with Ants={num_ants}, Evap={evap_rate}, Q={Q_val} in {gens} generations (Repeat {r+1}/{repeats}):")
                        print(final_board)
                        if best_overall_board is None or fitness(final_board, N) < best_overall_fitness:
                            best_overall_board = final_board
                            best_overall_fitness = fitness(final_board, N)
                    elif final_board is not None: # If not solved, but a board was returned
                        current_run_fitness = fitness(final_board, N)
                        if current_run_fitness < best_overall_fitness:
                            best_overall_fitness = current_run_fitness
                            best_overall_board = final_board


                avg_generations_solved = round(total_generations_for_solved / success_count, 2) if success_count > 0 else np.nan

                results.append({
                    "Lentelės dydis": f"{N}x{N}",
                    "Skruzdžių skaičius": num_ants,
                    "Garavimo greitis": evap_rate,
                    "Pheromone Q": Q_val,
                    "Sėkmės dažnis": f"{success_count}/{repeats}",
                    "Vid. generacijos (sėkm.)": avg_generations_solved # NEW COLUMN
                })

                # Print best board if no solution was found and best_overall_board exists
                if success_count == 0 and best_overall_board is not None:
                    print(f"For N={N}, Ants={num_ants}, Evap={evap_rate}, Q={Q_val}: No solution found in any repeat. Best board found (fitness {best_overall_fitness}):\n")
                    print(best_overall_board)
    return pd.DataFrame(results)

In [ ]:
# Parameters for ACO 4x4 Sudoku
num_ants_aco_4x4 = [10, 20]
evaporation_rates_aco_4x4 = [0.1, 0.3]
Q_values_aco_4x4 = [0.5, 1.0]
max_generations_aco_4x4 = 500

# Parameters for ACO 9x9 Sudoku
num_ants_aco_9x9 = [20, 50]
evaporation_rates_aco_9x9 = [0.1, 0.3]
Q_values_aco_9x9 = [0.5, 1.0]
max_generations_aco_9x9 = 1000

repeats_aco = 10 # Number of times to repeat each experiment, now 10

print("\n--- Running Ant Colony Optimization Experiments ---\n")

# Convert sudoku_4x4 and sudoku_9x9 to numpy arrays
sudoku_4x4_np = np.array(sudoku_4x4)
sudoku_9x9_np = np.array(sudoku_9x9)

df_aco_4x4 = run_experiment_aco(sudoku_4x4_np, 4, num_ants_aco_4x4, evaporation_rates_aco_4x4, Q_values_aco_4x4, repeats_aco, max_generations_aco_4x4)
df_aco_9x9 = run_experiment_aco(sudoku_9x9_np, 9, num_ants_aco_9x9, evaporation_rates_aco_9x9, Q_values_aco_9x9, repeats_aco, max_generations_aco_9x9)

# Display individual dataframes as they are now (before filtering)
display(df_aco_4x4)
display(df_aco_9x9)